# 第七章：生产实践与完整项目

## 学习目标
- 使用 LlamaIndex 内置评估器评估 RAG 质量
- 实现回调与可观测性调试
- 构建端到端的多源问答系统
- 理解 LlamaIndex 与 LangChain 的互操作
- 掌握框架选择的决策指南

## 0. 环境初始化

复用前几章的配置：加载环境变量、初始化 LLM 和 Embedding 模型、设置全局 Settings。

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.openai_like import OpenAILikeEmbedding
from llama_index.core import Settings

# 初始化 LLM
llm = OpenAILike(
    model="qwen-plus",
    api_base=os.environ["Qwen_API_BASE"],
    api_key=os.environ["Qwen_API_KEY"],
    is_chat_model=True,
)

# 切换为 GLM（Anthropic 兼容接口）：
# from llama_index.llms.anthropic import Anthropic
# llm = Anthropic(model="glm-4-plus", base_url=os.environ["GLM_API_BASE"], api_key=os.environ["GLM_API_KEY"])

# 初始化 Embedding 模型
embed_model = OpenAILikeEmbedding(
    model_name="text-embedding-v3",
    api_base=os.environ["Qwen_API_BASE"],
    api_key=os.environ["Qwen_API_KEY"],
)

# 全局配置
Settings.llm = llm
Settings.embed_model = embed_model

print(f"LLM: {Settings.llm.model}")
print(f"Embedding: {Settings.embed_model.model_name}")

## 1. RAG 评估

构建 RAG 系统后，一个关键问题是：**你怎么知道它回答得好不好？**

仅靠人工抽查不可扩展，我们需要自动化的评估指标。LlamaIndex 内置了评估器，可以从多个维度衡量 RAG 质量。

In [ ]:
from llama_index.core import VectorStoreIndex, Document

# 准备知识库
documents = [
    Document(
        text="LlamaIndex 是一个专注于数据连接和检索的 LLM 框架。它的核心能力是将各种数据源（文件、数据库、API）与 LLM 连接起来，提供高质量的检索增强生成（RAG）功能。",
        metadata={"source": "overview"},
    ),
    Document(
        text="LlamaIndex 的查询引擎由三个组件组成：Retriever 负责检索相关节点，Node Postprocessor 负责过滤和重排序，Response Synthesizer 负责生成最终回答。",
        metadata={"source": "architecture"},
    ),
    Document(
        text="LlamaIndex 支持多种向量存储后端，包括 FAISS、Pinecone、Weaviate、Chroma 等。通过 StorageContext 统一管理存储，支持持久化到磁盘。",
        metadata={"source": "storage"},
    ),
    Document(
        text="LlamaIndex 的高级检索策略包括混合检索（向量 + BM25）、HyDE 查询变换、子问题分解等，这些是提升 RAG 质量的关键手段。",
        metadata={"source": "retrieval"},
    ),
]

index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=2)
print(f"索引构建完成，包含 {len(documents)} 个文档")

In [ ]:
from llama_index.core.evaluation import FaithfulnessEvaluator, RelevancyEvaluator

# 忠实度评估：回答是否基于检索到的内容
faithfulness_evaluator = FaithfulnessEvaluator()

# 相关性评估：检索内容是否与问题相关
relevancy_evaluator = RelevancyEvaluator()

# 执行查询
query = "LlamaIndex 的查询引擎由哪些组件组成？"
response = query_engine.query(query)
print("回答:", response)

In [ ]:
# 评估忠实度
faith_result = faithfulness_evaluator.evaluate_response(response=response)
print(f"忠实度: {'通过' if faith_result.passing else '未通过'}")
print(f"分数: {faith_result.score}")

# 评估相关性
rel_result = relevancy_evaluator.evaluate_response(query=query, response=response)
print(f"相关性: {'通过' if rel_result.passing else '未通过'}")
print(f"分数: {rel_result.score}")

### 刚才发生了什么？

1. **`FaithfulnessEvaluator`** 检查回答是否「忠实」于检索到的内容——即模型有没有编造检索结果中不存在的信息。这是检测幻觉的核心指标。
2. **`RelevancyEvaluator`** 检查检索到的内容是否与用户问题相关——如果检索结果都是无关内容，再好的 LLM 也答不好。
3. 两个评估器内部都会调用 LLM 来判断，所以评估本身也消耗 token。

### 评估指标说明

| 评估维度 | 说明 | 评估器 |
|---------|------|--------|
| 忠实度（Faithfulness） | 回答是否基于检索到的内容，不编造 | `FaithfulnessEvaluator` |
| 相关性（Relevancy） | 检索内容是否与问题相关 | `RelevancyEvaluator` |

### 与 LangChain 对比

LangChain 本身**没有内置**评估器。要评估 RAG 质量，通常需要借助外部工具：
- **LangSmith**：LangChain 官方的可观测性平台（需要注册账号、联网上报）
- **ragas**：第三方评估库，提供类似的忠实度、相关性指标

LlamaIndex 的优势是评估器**开箱即用**，不需要额外注册服务或安装第三方库。

### 1.1 批量评估

单条评估只能看个案，批量评估才能了解系统整体质量。

In [ ]:
# 批量评估
test_questions = [
    "LlamaIndex 是什么？",
    "LlamaIndex 支持哪些向量存储？",
    "如何提升 RAG 质量？",
]

print("批量评估结果：")
print(f"{'问题':<30} {'忠实度':<8} {'相关性':<8}")
print("-" * 46)

for q in test_questions:
    response = query_engine.query(q)
    faith = faithfulness_evaluator.evaluate_response(response=response)
    rel = relevancy_evaluator.evaluate_response(query=q, response=response)
    print(f"{q:<30} {faith.score:<8} {rel.score:<8}")

### 刚才发生了什么？

我们对 3 个测试问题逐一评估了忠实度和相关性。在生产环境中，你应该：

1. **准备一个标准测试集**（20~50 个问题），覆盖不同类型的查询
2. **每次修改 RAG 管线后重新评估**，确保改进没有引入退化
3. **关注分数低的问题**，分析是检索问题还是合成问题

### 常见问题

**Q：评估分数都是 1.0 或 0.0，没有中间值？**

LlamaIndex 的内置评估器默认输出二值结果（通过/未通过）。如果需要更细粒度的分数，可以自定义评估提示词，让 LLM 输出 1-5 分的评分。

**Q：评估消耗太多 token 怎么办？**

每次评估会调用 LLM，批量评估的 token 消耗可能比查询本身还高。建议：
- 评估时使用较便宜的模型
- 不要每次查询都评估，定期采样评估即可

## 2. 回调与可观测性

当 RAG 系统的回答不理想时，你需要知道**哪个环节**出了问题：
- 是检索没找到正确的文档？
- 还是 LLM 合成回答时出了错？

LlamaIndex 的回调系统可以追踪整个查询流程的每一步。

In [ ]:
from llama_index.core.callbacks import CallbackManager, LlamaDebugHandler

# 设置调试回调
llama_debug = LlamaDebugHandler(print_trace_on_end=True)
callback_manager = CallbackManager([llama_debug])

# 将回调管理器设置到全局 Settings
Settings.callback_manager = callback_manager

# 重建查询引擎（使回调生效）
query_engine_debug = index.as_query_engine(similarity_top_k=2)

# 执行查询（会输出详细的执行追踪）
response = query_engine_debug.query("LlamaIndex 的存储架构是什么？")
print("\n最终回答:", response)

### 刚才发生了什么？

1. **`LlamaDebugHandler`** 是 LlamaIndex 内置的调试处理器，设置 `print_trace_on_end=True` 会在每次查询结束后打印完整的执行追踪。
2. **`CallbackManager`** 管理所有回调处理器，可以同时添加多个。
3. 通过 `Settings.callback_manager` 全局设置后，所有后续创建的查询引擎都会自动使用这个回调管理器。

追踪信息包括：
- 每个组件的调用顺序和耗时
- Embedding 生成的时间
- LLM 调用的输入输出
- 检索到的节点信息

### 与 LangChain 对比

| | LangChain | LlamaIndex |
|--|-----------|------------|
| 基础调试 | `set_debug(True)` 打印详细日志 | `LlamaDebugHandler` 打印执行追踪 |
| 可观测性平台 | LangSmith（云端，需注册） | 支持多种后端（Arize Phoenix、OpenLLMetry 等） |
| 回调机制 | `callbacks` 参数 / `RunnableConfig` | `CallbackManager` + 全局 `Settings` |
| 自定义程度 | 自定义 `BaseCallbackHandler` | 自定义 `BaseCallbackHandler` |

LlamaIndex 的回调系统更集中化（通过 Settings 全局管理），LangChain 更分散（每个 Runnable 可以有自己的 callbacks）。

In [ ]:
# 重置回调管理器（避免后续输出过多调试信息）
Settings.callback_manager = CallbackManager()

### 2.1 流式输出：提升用户体验

在生产环境中，流式输出是标配。用户不愿意等 5 秒看到一整段回答，更希望看到文字逐步出现。

In [ ]:
# 流式输出
engine = index.as_query_engine(streaming=True, similarity_top_k=2)
response = engine.query("总结 LlamaIndex 的核心特性")

print("流式输出：")
for text in response.response_gen:
    print(text, end="", flush=True)
print()

### 刚才发生了什么？

1. `streaming=True` 启用流式模式
2. `response.response_gen` 是一个生成器，逐步产出文本片段
3. 在 Web 应用中，可以将这些片段通过 Server-Sent Events（SSE）推送给前端

### 常见问题

**Q：流式模式下如何同时获取 source_nodes？**

`response.source_nodes` 在流式模式下依然可用，因为检索步骤在 LLM 开始生成之前就完成了。你可以在消费 `response_gen` 之前就读取 `source_nodes`。

## 3. 完整项目：多源问答系统

现在把前面学到的知识整合起来，构建一个端到端的多源问答系统。

系统架构：
```
用户问题
   │
   ▼
┌──────────────────────┐
│  RouterQueryEngine    │  根据问题内容自动选择知识领域
│  （路由查询引擎）       │
└──────────────────────┘
   │         │
   ▼         ▼
┌────────┐ ┌────────┐
│ AI 知识 │ │ Web 知识│
│   库    │ │   库    │
└────────┘ └────────┘
```

In [ ]:
# === 第一步：准备多个知识领域的文档 ===

ai_docs = [
    Document(
        text="深度学习是机器学习的一个分支，使用多层神经网络来学习数据的层次化表示。常见的深度学习架构包括 CNN（卷积神经网络，用于图像）、RNN（循环神经网络，用于序列）和 Transformer（用于自然语言处理）。",
        metadata={"domain": "AI", "topic": "深度学习"},
    ),
    Document(
        text="Transformer 架构由 Google 在 2017 年提出，核心是自注意力机制（Self-Attention）。它摒弃了 RNN 的顺序处理方式，可以并行处理序列中的所有位置，大幅提升了训练效率。GPT、BERT、LLaMA 等模型都基于 Transformer。",
        metadata={"domain": "AI", "topic": "Transformer"},
    ),
    Document(
        text="训练神经网络的核心步骤：1) 前向传播计算预测值；2) 用损失函数计算误差；3) 反向传播计算梯度；4) 用优化器（如 Adam）更新参数。关键超参数包括学习率、批次大小和训练轮数。",
        metadata={"domain": "AI", "topic": "训练"},
    ),
    Document(
        text="RAG（检索增强生成）是将信息检索与文本生成结合的技术。它先从外部知识库检索相关文档，再将检索结果作为上下文提供给 LLM，让模型基于真实数据回答问题，有效减少幻觉。",
        metadata={"domain": "AI", "topic": "RAG"},
    ),
]

web_docs = [
    Document(
        text="React 是 Meta 开发的前端 JavaScript 库，采用组件化和虚拟 DOM 设计。它使用 JSX 语法将 HTML 和 JavaScript 混合编写，通过单向数据流管理状态。React 生态丰富，有 Next.js（SSR）、React Native（移动端）等框架。",
        metadata={"domain": "Web", "topic": "React"},
    ),
    Document(
        text="Vue 是尤雨溪创建的渐进式前端框架。它的核心特点是易学易用，采用模板语法和响应式数据绑定。Vue 3 引入了 Composition API，提供了更灵活的代码组织方式。配套工具包括 Vite（构建工具）、Pinia（状态管理）、Vue Router（路由）。",
        metadata={"domain": "Web", "topic": "Vue"},
    ),
    Document(
        text="RESTful API 是一种基于 HTTP 协议的 Web 服务设计风格。核心原则包括：资源用 URL 标识、用 HTTP 方法表达操作（GET/POST/PUT/DELETE）、无状态、返回 JSON 格式数据。FastAPI 和 Express 是常用的后端框架。",
        metadata={"domain": "Web", "topic": "API"},
    ),
    Document(
        text="WebSocket 是一种在单个 TCP 连接上进行全双工通信的协议。与 HTTP 的请求-响应模式不同，WebSocket 允许服务端主动向客户端推送消息，适用于实时聊天、在线协作、股票行情等场景。",
        metadata={"domain": "Web", "topic": "WebSocket"},
    ),
]

print(f"AI 领域文档: {len(ai_docs)} 篇")
print(f"Web 领域文档: {len(web_docs)} 篇")

In [ ]:
# === 第二步：为每个领域构建独立索引 ===

ai_index = VectorStoreIndex.from_documents(ai_docs)
web_index = VectorStoreIndex.from_documents(web_docs)

print("AI 知识索引构建完成")
print("Web 知识索引构建完成")

In [ ]:
# === 第三步：将索引包装为查询工具 ===
from llama_index.core.tools import QueryEngineTool, ToolMetadata

tools = [
    QueryEngineTool(
        query_engine=ai_index.as_query_engine(similarity_top_k=2),
        metadata=ToolMetadata(
            name="ai_knowledge",
            description="AI 和机器学习相关知识，包括深度学习、Transformer、神经网络训练、RAG 等",
        ),
    ),
    QueryEngineTool(
        query_engine=web_index.as_query_engine(similarity_top_k=2),
        metadata=ToolMetadata(
            name="web_knowledge",
            description="Web 开发相关知识，包括 React、Vue、RESTful API、WebSocket 等",
        ),
    ),
]

print(f"创建了 {len(tools)} 个查询工具：")
for tool in tools:
    print(f"  - {tool.metadata.name}: {tool.metadata.description}")

### 刚才发生了什么？

`QueryEngineTool` 将查询引擎包装成「工具」，每个工具有名称和描述。路由引擎会根据这些描述来判断应该将问题发送给哪个工具。

这和 LangChain Agent 中的 `Tool` 概念类似——都是通过自然语言描述让 LLM 选择合适的工具。区别在于 LlamaIndex 的工具专门为查询引擎设计，集成更紧密。

In [ ]:
# === 第四步：构建路由查询引擎 ===
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

router = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=tools,
)

print("路由查询引擎构建完成")

### 刚才发生了什么？

`RouterQueryEngine` 是 LlamaIndex 的路由引擎，它：
1. 接收用户问题
2. 用 `LLMSingleSelector` 让 LLM 根据工具描述选择最合适的查询引擎
3. 将问题转发给选中的引擎
4. 返回该引擎的回答

`LLMSingleSelector` 表示每次只选一个工具。如果问题涉及多个领域，还可以用 `LLMMultiSelector` 同时查询多个工具。

In [ ]:
# === 第五步：测试多源路由 ===
questions = [
    "什么是深度学习？",
    "React 和 Vue 有什么区别？",
    "如何训练一个神经网络？",
    "WebSocket 适用于什么场景？",
]

for q in questions:
    print(f"Q: {q}")
    response = router.query(q)
    # 显示路由选择了哪个工具
    source_nodes = response.source_nodes
    if source_nodes:
        domain = source_nodes[0].metadata.get("domain", "未知")
        print(f"   [路由到: {domain} 知识库]")
    print(f"A: {response}")
    print("-" * 60)

### 刚才发生了什么？

路由引擎自动将 AI 相关的问题（深度学习、神经网络训练）发送到 AI 知识库，将 Web 相关的问题（React vs Vue、WebSocket）发送到 Web 知识库。

这就是多源问答系统的核心：**用户不需要知道知识存放在哪里，系统自动路由到正确的数据源**。

### 常见问题

**Q：如果问题跨越两个领域怎么办？**

比如「如何用 React 构建一个 RAG 应用的前端？」这种问题同时涉及 Web 和 AI。可以用 `LLMMultiSelector` 替换 `LLMSingleSelector`，它会选择多个工具并合并结果。

**Q：路由选错了怎么办？**

优化工具的 `description`。描述越精确，LLM 路由越准确。避免模糊描述如「通用知识」，改用具体的关键词列表。

## 4. 与 LangChain 互操作

LlamaIndex 和 LangChain 不是互斥的。在生产项目中，常见的模式是：
- **LlamaIndex 负责数据连接和检索**（它更擅长这个）
- **LangChain / LangGraph 负责编排和 Agent 逻辑**（它更擅长这个）

LlamaIndex 提供了 `as_langchain_tool()` 方法，可以将查询引擎直接转换为 LangChain 工具。

In [ ]:
# LlamaIndex 查询引擎作为 LangChain 工具
from llama_index.core.tools import QueryEngineTool, ToolMetadata

llamaindex_tool = QueryEngineTool(
    query_engine=index.as_query_engine(),
    metadata=ToolMetadata(
        name="knowledge_base",
        description="知识库查询工具，可以回答关于 LlamaIndex 框架的问题",
    ),
)

# 转换为 LangChain 工具
langchain_tool = llamaindex_tool.as_langchain_tool()
print(f"LangChain 工具名: {langchain_tool.name}")
print(f"LangChain 工具描述: {langchain_tool.description}")

### 刚才发生了什么？

`as_langchain_tool()` 将 LlamaIndex 的 `QueryEngineTool` 转换为 LangChain 的 `Tool` 对象。转换后的工具可以直接用在 LangChain Agent 或 LangGraph 工作流中。

典型的互操作架构：

```
┌─────────────────────────────────────┐
│        LangGraph 工作流               │
│                                     │
│  ┌───────────┐  ┌───────────────┐  │
│  │ LlamaIndex │  │ 其他 LangChain │  │
│  │ 查询工具   │  │ 工具（搜索等） │  │
│  └───────────┘  └───────────────┘  │
│         │              │            │
│         └──────┬───────┘            │
│                ▼                    │
│         Agent 决策节点               │
└─────────────────────────────────────┘
```

> **实际建议**：如果你的项目主要是文档问答，直接用 LlamaIndex 就够了。如果需要多步骤推理、循环决策、多 Agent 协作，再引入 LangGraph 做编排，LlamaIndex 负责 RAG 部分。

In [ ]:
# 验证转换后的工具可以正常调用
result = langchain_tool.invoke("LlamaIndex 的查询引擎有哪些组件？")
print("通过 LangChain 工具调用 LlamaIndex：")
print(result)

### 刚才发生了什么？

我们用 LangChain 的 `.invoke()` 接口调用了一个底层是 LlamaIndex 查询引擎的工具。整个流程对 LangChain 来说是透明的——它不知道（也不需要知道）背后用的是 LlamaIndex。

### 常见问题

**Q：互操作有性能开销吗？**

几乎没有。`as_langchain_tool()` 只是一层薄封装，核心的检索和合成逻辑仍然在 LlamaIndex 内部执行。

**Q：流式输出在互操作时支持吗？**

通过 `as_langchain_tool()` 转换后，流式输出的支持取决于 LangChain 侧的调用方式。如果需要流式，建议在 LlamaIndex 侧直接使用流式查询引擎。

## 5. 框架选择指南

学完了 LangChain、LangGraph 和 LlamaIndex，面对实际项目时该选哪个？以下是基于场景的决策指南。

| 场景 | 推荐框架 | 原因 |
|------|---------|------|
| 文档问答 / 知识库检索 | LlamaIndex | 内置索引和查询引擎，开箱即用 |
| 复杂 Agent / 多步骤工作流 | LangGraph | 图结构支持循环和条件分支 |
| 快速原型 / 简单链 | LangChain LCEL | 管道语法简洁 |
| 高级检索策略 | LlamaIndex | 内置混合检索、重排序、HyDE 等 |
| 多 Agent 协作 | LangGraph | Supervisor 模式、子图 |
| 生产级 RAG | LlamaIndex + LangGraph | LlamaIndex 做检索，LangGraph 做编排 |

### 决策流程

```
你的需求是什么？
   │
   ├─ 文档问答 / 知识库检索
   │     └─→ LlamaIndex（开箱即用的 RAG）
   │
   ├─ 多步骤 Agent / 复杂工作流
   │     └─→ LangGraph（图结构编排）
   │
   ├─ 简单的 LLM 调用链
   │     └─→ LangChain LCEL（管道语法）
   │
   └─ 生产级系统（RAG + Agent）
         └─→ LlamaIndex（检索）+ LangGraph（编排）
```

> **一句话总结**：LlamaIndex 是 RAG 专家，LangGraph 是编排专家，LangChain 是通用工具箱。三者可以组合使用，不必只选一个。

## 6. 学习路线回顾

回顾 LlamaIndex 教程的 7 个章节，看看我们走过了怎样的学习路径。

| 章节 | 核心知识 | 对应 LangChain 概念 |
|------|---------|-------------------|
| 01 环境搭建与核心概念 | LLM、Document、Settings | ChatOpenAI、Document、无 |
| 02 文档加载与索引构建 | SimpleDirectoryReader、VectorStoreIndex | TextLoader、FAISS |
| 03 查询引擎与响应合成 | QueryEngine、响应合成模式 | RAG Chain、LCEL |
| 04 向量存储与持久化 | FAISS 持久化、StorageContext | FAISS save_local/load_local |
| 05 高级检索策略 | 混合检索、HyDE、子问题分解 | EnsembleRetriever（有限） |
| 06 对话引擎与路由 | ChatEngine、RouterQueryEngine | Memory、Agent |
| 07 生产实践与完整项目 | 评估、可观测性、互操作 | LangSmith、ragas |

### LlamaIndex 编程模型全貌

```
数据层                    索引层                    查询层                    应用层
─────                    ─────                    ─────                    ─────
Document          →   VectorStoreIndex    →   QueryEngine          →   RouterQueryEngine
SimpleDirectoryReader     StorageContext          Retriever                ChatEngine
Node/TextNode             FAISS/Chroma            ResponseSynthesizer      评估 & 可观测性
                                                  NodePostprocessor        LangChain 互操作
```

从左到右，就是一个完整的 LlamaIndex RAG 系统的构建过程。

## 总结

本章学习了：
- FaithfulnessEvaluator 和 RelevancyEvaluator 评估 RAG 质量
- CallbackManager 和 LlamaDebugHandler 调试查询流程
- 构建完整的多源路由问答系统
- LlamaIndex 与 LangChain 的互操作（as_langchain_tool）
- 框架选择决策指南

至此，LlamaIndex 教程全部完成。你已经掌握了从环境搭建到生产部署的完整知识链路。

## 进阶方向

- **LlamaIndex Workflows**：声明式、事件驱动的编排系统
- **自定义 Retriever**：实现特定领域的检索逻辑
- **多模态索引**：支持图片、表格等非文本数据
- **与 Neo4j 结合**：构建知识图谱增强的 RAG 系统（见 neo4j/ 教程）